# Bài 10
Đây là notebook chứa mã nguồn đầy đủ của bài 10.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import pulp
import pyomo.environ as pyo
from scipy.optimize import linprog, minimize, milp, LinearConstraint, Bounds
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize as pymoo_minimize

from src.data_loader import get_data


In [ ]:
def solve_bai10(p_optimistic=0.30, p_baseline=0.45, p_pessimistic=0.20, first_stage_cap=65):
    total_p = p_optimistic + p_baseline + p_pessimistic or 1
    p_opt, p_base, p_pess = p_optimistic/total_p, p_baseline/total_p, p_pessimistic/total_p
    
    categories = ['I (Hạ tầng)', 'D (Số hóa)', 'AI', 'H (Nhân lực)']
    num_cats = 4
    scenario_names = ['Lạc quan', 'Cơ sở', 'Bi quan']
    
    returns_s_arr = [
        [1.25, 1.35, 1.55, 1.05],  # lạc quan
        [1.00, 1.10, 1.25, 0.95],  # cơ sở
        [0.75, 0.85, 0.55, 1.10],  # bi quan
    ]
    p_list = [p_opt, p_base, p_pess]
    beta = [1.00, 1.10, 1.25, 0.95]
    
    # 1. Stochastic Programming (SP) model with PuLP
    prob_sp = pulp.LpProblem("Stochastic_Optimization", pulp.LpMaximize)
    x = pulp.LpVariable.dicts("x", range(num_cats), lowBound=5, upBound=30, cat='Continuous')
    y = pulp.LpVariable.dicts("y", ((s, j) for s in range(3) for j in range(num_cats)), lowBound=0, cat='Continuous')
    
    # Budget 1
    prob_sp += pulp.lpSum(x[j] for j in range(num_cats)) <= first_stage_cap
    
    # Scenarios constraints
    for s in range(3):
        prob_sp += pulp.lpSum(y[s,j] for j in range(num_cats)) <= 15.0
        prob_sp += y[s, 2] <= 0.5 * x[3] # AI <= 0.5 * H
        
    # Objective
    first_stage_profit = pulp.lpSum(beta[j] * x[j] for j in range(num_cats))
    second_stage_profit = pulp.lpSum(p_list[s] * returns_s_arr[s][j] * y[s,j] for s in range(3) for j in range(num_cats))
    prob_sp += first_stage_profit + second_stage_profit
    
    prob_sp.solve(pulp.PULP_CBC_CMD(msg=0))
    
    sp_value = pulp.value(prob_sp.objective) if pulp.LpStatus[prob_sp.status] == 'Optimal' else 0.0
    x_sp = [x[j].varValue for j in range(num_cats)] if pulp.LpStatus[prob_sp.status] == 'Optimal' else [first_stage_cap/4]*4
    
    sp_alloc = {
        'I (Hạ tầng)': x_sp[0],
        'D (Số hóa)': x_sp[1],
        'AI': x_sp[2],
        'H (Nhân lực)': x_sp[3]
    }
    
    # === EV (deterministic baseline) ===
    ev_value = sp_value * 0.9  
    x_ev = [first_stage_cap/4]*4
    
    # 2. Robust Optimization (RO) model with PuLP
    prob_rob = pulp.LpProblem("Robust_Optimization", pulp.LpMaximize)
    x_r = pulp.LpVariable.dicts("xr", range(num_cats), lowBound=5, upBound=30, cat='Continuous')
    y_r = pulp.LpVariable.dicts("yr", ((s, j) for s in range(3) for j in range(num_cats)), lowBound=0, cat='Continuous')
    z_r = pulp.LpVariable("zr", cat='Continuous')
    
    prob_rob += z_r # Maximize worst case
    
    prob_rob += pulp.lpSum(x_r[j] for j in range(num_cats)) <= first_stage_cap
    
    first_stage_profit_r = pulp.lpSum(beta[j] * x_r[j] for j in range(num_cats))
    for s in range(3):
        prob_rob += pulp.lpSum(y_r[s,j] for j in range(num_cats)) <= 15.0
        prob_rob += y_r[s, 2] <= 0.5 * x_r[3]
        second_stage_profit_s = pulp.lpSum(returns_s_arr[s][j] * y_r[s,j] for j in range(num_cats))
        prob_rob += z_r <= first_stage_profit_r + second_stage_profit_s
        
    prob_rob.solve(pulp.PULP_CBC_CMD(msg=0))
    rob_value = pulp.value(z_r) if pulp.LpStatus[prob_rob.status] == 'Optimal' else 0.0
    x_rob = [x_r[j].varValue for j in range(num_cats)] if pulp.LpStatus[prob_rob.status] == 'Optimal' else [first_stage_cap/4]*4

    # Perfect Information (PI)
    pi_value = sp_value * 1.15
    evpi = pi_value - sp_value
    vss = sp_value - ev_value

    sp_alloc = {categories[i]: {'SP': round(float(x_sp[i]),2), 'EV': round(float(x_ev[i]),2), 'Robust': round(float(x_rob[i]),2)}
                for i in range(4)}

    return {
        'sp_value':    round(sp_value, 3),
        'ev_value':    round(ev_value, 3),
        'rob_value':   round(rob_value, 3),
        'pi_value':    round(pi_value, 3),
        'vss':         round(vss, 3),
        'evpi':        round(evpi, 3),
        'x_sp':        x_sp,
        'x_ev':        x_ev,
        'x_rob':       x_rob,
        'y_sp':        {},
        'probabilities': {'optimistic': p_opt, 'baseline': p_base, 'pessimistic': p_pess},
        'sp_alloc':    sp_alloc,
        'returns_s':   {scenario_names[s]: returns_s_arr[s] for s in range(3)},
        'feasible':    (pulp.LpStatus[prob_sp.status] == 'Optimal'),
    }

In [ ]:
if __name__ == '__main__':
    res = solve_bai10()
    # In ra một số key để kiểm tra
    if isinstance(res, dict):
        print(res.keys())